# 04 — Deepfake detection

**Status: this model is not trained.** FaceForensics++ requires a signed
agreement and the fine-tune requires a GPU. What exists is the complete
pipeline, the split discipline, and the honest null path.

This notebook documents the two decisions that determine whether any future
number here means anything.

## 1. Split by source video, never by frame

Frames from one video in both train and test is *the* mechanism behind
deepfake papers reporting 99% accuracy. Adjacent frames of one clip are
near-identical images; a model that memorizes a face scores perfectly on the
rest of that clip and fails on everything else.

`tests/test_splits.py` asserts that a frame-level split raises. The cell
below demonstrates it.

## 2. 'No face found' and 'real face' are different answers

When no face is detected, the contract gets `face_detected=false` and
`deepfake_prob=null` — never a low score. A low score says 'we looked and it
seems real'; a null says 'we could not look'. Conflating them means a
deepfake checker that quietly clears every clip it failed to parse.

In [ ]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from modeling.config import get_settings, run_fingerprint, set_all_seeds
from modeling.io import CorpusReader, ScoredStore

set_all_seeds()
settings = get_settings()
reader = CorpusReader(settings)
store = ScoredStore(settings)

# Every number below is tied to this fingerprint. If a rerun disagrees with
# a committed figure, this block is where the diagnosis starts.
fingerprint = run_fingerprint()
print(f"seed={fingerprint['seed']}  device={fingerprint['device']}  "
      f"corpus={fingerprint['input_manifest_hash']}")

In [ ]:
import pytest
from modeling.datasets.splits import LeakageError, assert_no_group_leakage

frames = pd.DataFrame([
    {'frame_id': f'{video}-f{i:03d}', 'video_id': video}
    for video in ('v001_real', 'v001_fake', 'v002_real', 'v002_fake')
    for i in range(50)
])
rng = np.random.default_rng(7)
order = rng.permutation(len(frames))

try:
    assert_no_group_leakage(frames['video_id'], order[:150], order[150:])
    print('NO ERROR RAISED — the leakage guard is broken')
except LeakageError as exc:
    print('correctly refused a frame-level split:')
    print(' ', exc)

## The dataset loader

FF++ names manipulated clips `<target>_<source>.mp4`. The loader groups on
the **target** identity, because the manipulated clip shares that person's
face, background and lighting with `original_sequences/<target>`. Clips whose
filename cannot be parsed are dropped rather than grouped by guess.

DFDC is used as a cross-dataset generalisation check, not as training data:
it has no per-method labels, and the question that matters for a public
upload checker is whether the model transfers to production pipelines it has
never seen.

In [ ]:
from modeling.datasets import get_dataset
from modeling.datasets.splits import domain_holdout, group_train_val_test

dataset = get_dataset('faceforensics')
on_disk = dataset.available()
print('FaceForensics++ on disk:', on_disk)
loaded = dataset.load(demo=not on_disk)
print(loaded.summary())

In [ ]:
work, split = group_train_val_test(loaded.frame, group_col='source_video',
                                   label_col='label', text_col='video_id',
                                   dedupe=False)
print(split.describe())

# Every metric this model ever reports must carry that line.
for name, index in (('train', split.train), ('val', split.val), ('test', split.test)):
    videos = set(work['source_video'].iloc[index])
    print(f'{name:6} {len(index):4} clips, {len(videos):3} source videos')

## Cross-method generalisation

The known weak point of this entire model family. Train on three FF++
manipulation methods, test on the held-out fourth. These numbers will be
worse than the in-method ones — reporting them is a strength, not a
weakness, and a reviewer who does not see them should assume the worst.

In [ ]:
for held_out in ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures']:
    try:
        w, s = domain_holdout(loaded.frame, domain_col='method',
                              held_out=held_out, group_col='source_video')
        print(f'hold out {held_out:15} train={len(s.train):4} test={len(s.test):4}')
    except ValueError as exc:
        print(f'hold out {held_out:15} unavailable: {exc}')

## CPU inference latency

A deployment fact, not a curiosity: Phase 4 runs this on demand for uploads
and needs to know what it costs. Xception over 16 frames is seconds per video
on CPU, which is acceptable for an on-demand upload path and not acceptable
for scoring a whole corpus — which is why corpus-wide media scores are
precomputed into Parquet instead.

In [ ]:
from modeling.media.frames import FrameExtractor, describe

extractor = FrameExtractor(settings, detector='none')
sample = sorted(Path(loaded.frame['path'].iloc[0]).parent.glob('*'))[:3]
for path in sample:
    import time
    start = time.perf_counter()
    extraction = extractor.extract(path)
    elapsed = (time.perf_counter() - start) * 1000
    print(f'{path.name:20} {elapsed:6.1f} ms  {describe(extraction)[:90]}')